In [2]:
!pip install pyspark delta-spark

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("DeltaLakeFullDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.7 MB/s eta 0:00:00


In [3]:
student_data = [
    (101, "Aman", 85),
    (102, "Sita", 92),
    (103, "Rahul", 78)
]
columns = ["id", "name", "marks"]

#creating dataframe
student_df = spark.createDataFrame(student_data, columns)

#writing the data into Delta format
student_df.write.format("delta").mode("overwrite").save("/delta/student_marks")

#reading the Delta table
read_df = spark.read.format("delta").load("/delta/student_marks")
read_df.show()

+---+-----+-----+
| id| name|marks|
+---+-----+-----+
|102| Sita|   92|
|103|Rahul|   78|
|101| Aman|   85|
+---+-----+-----+



TASK 2 :

In [7]:
emp_data = [
    (1, "Alice", 50000),
    (2, "Bob", 60000)
]

emp_cols = ["id", "name", "salary"]
emp_df = spark.createDataFrame(emp_data, emp_cols)
emp_df.show()

+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice| 50000|
|  2|  Bob| 60000|
+---+-----+------+



In [8]:
path = "/tmp/employee_salary"
emp_df.write.format("delta").mode("overwrite").save(path)

In [9]:
new_emps = [(3, "Charlie", 55000), (4, "David", 48000)]
new_emp_df = spark.createDataFrame(new_emps, emp_cols)
new_emp_df.write.format("delta").mode("append").save(path)

In [10]:
# 4. Update salary of one employee (increase by 10%)
# We use the DeltaTable object to perform DML operations
delta_table = DeltaTable.forPath(spark, path)

delta_table.update(
    condition="name = 'Bob'",
    set={"salary": "salary * 1.10"}
)

# 5. Display final table
delta_table.toDF().show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  3|Charlie| 55000|
|  4|  David| 48000|
|  1|  Alice| 50000|
|  2|    Bob| 66000|
+---+-------+------+



TASK 3:

In [11]:
from delta.tables import DeltaTable

# 1. Create initial dataset
initial_sales = [
    (1, "Amit", "Laptop", 50000),
    (2, "Riya", "Phone", 22000),
    (3, "John", "Tablet", 18000)
]
columns = ["id", "name", "product", "sales"]
df = spark.createDataFrame(initial_sales, columns)

In [12]:
path = "/tmp/company_sales_pipeline"
df.write.format("delta").mode("overwrite").save(path)

In [13]:
new_sales = [(4, "Sara", "Camera", 35000)]
new_df = spark.createDataFrame(new_sales, columns)
new_df.write.format("delta").mode("append").save(path)

In [14]:
delta_table = DeltaTable.forPath(spark, path)

In [15]:
delta_table.update(
    condition="name = 'Riya'",
    set={"sales": "sales + 3000"}
)

In [16]:
delta_table.delete("sales < 20000")

In [17]:
print("Final Data State:")
delta_table.toDF().show()

Final Data State:
+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  1|Amit| Laptop|50000|
|  4|Sara| Camera|35000|
|  2|Riya|  Phone|25000|
+---+----+-------+-----+



In [19]:
original_df = spark.read.format("delta")\
.option("versionAsOf", 0)\
.load(path)

original_df.show()

+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  2|Riya|  Phone|22000|
|  3|John| Tablet|18000|
|  1|Amit| Laptop|50000|
+---+----+-------+-----+

